Урок в прозе: https://proproprogs.ru/python_oop/python-nasledovanie-funkciya-super-i-delegirovanie

Телеграм-канал: https://t.me/python_selfedu

In [ ]:
class Geom:
    name = 'geom'

class Line(Geom): # это называется расширением базового класса
    def draw(self):
        print('рисование линии')

In [ ]:
class Geom:
    name = 'geom'

    def draw(self):
        print('рисование линии')

class Line(Geom): # а тут уже переопределение (overriding)
    def draw(self):
        print('рисование линии')

In [1]:
class Geom:
    name = 'geom'

    def __init__(self):
        print('инициализация Геом')

class Line(Geom):
    def draw(self):
        print('рисование линии')

l = Line()

инициализация Геом


Здесь произошла следующая цепочка вызовов

Из-за скруглых скобок вызывается метод __call__

__call__(self, *args, **kwargs):
    obj = self.__new__(self, *args, **kwargs)
    self.__init__(obj, *args, **kwargs)
    return obj

Он оследовательно вызывает методы __new__ (для создания экземпляра класса и потом __init__. Все эти методы в первую очередь ищутся в текущем классе Line. Если не находятся, то по цепочке идем в базовый класс. Так, __init__ найден в классе Geom, а __new__ только в классе object. Кстати (очень важно), несмотря на то, что __init__ найден в классе Geom, self в этом __init__будет ссылаться на экземпляр Line, потому что вызван из него

ну а если инициализатор будет найден сразу в классе Line, то он и будет выбран

In [4]:
class Geom:
    name = 'geom'

    def __init__(self):
        print('инициализация Геом')

class Line(Geom):

    def __init__(self, x1, y1, x2, y2):
        print('инициализация Line')
        self.x1 = x1
        self.y1 = y1
        self.x2 = x2
        self.y2 = y2

l = Line(1,2,3,4)
print(l.__dict__)

инициализация Line
{'x1': 1, 'y1': 2, 'x2': 3, 'y2': 4}


Представим, что нужно создать еще один класс, очень похожий

In [5]:
class Geom:
    name = 'geom'
    def __init__(self):
        print('инициализация Геом')

class Line(Geom):
    def __init__(self, x1, y1, x2, y2):
        print('инициализация Line')
        self.x1 = x1
        self.y1 = y1
        self.x2 = x2
        self.y2 = y2

class Rect(Geom):
    def __init__(self, x1, y1, x2, y2, fill=None):
        self.x1 = x1
        self.y1 = y1
        self.x2 = x2
        self.y2 = y2
        self.fill = fill

В классах получилось дублирование кода. Общее нужно вынести в класс Geom

In [9]:
class Geom:
    def __init__(self, x1, y1, x2, y2):
        print(f'инициализация Геом для {self.__class__}')
        self.x1 = x1
        self.y1 = y1
        self.x2 = x2
        self.y2 = y2

class Line(Geom):
    pass

class Rect(Geom):
    def __init__(self, x1, y1, x2, y2, fill=None):
        print('инициализатор Rect')
        self.fill = fill

l = Line(1,2,3,4) # вызыван инициализатор из базового класса
r = Rect(5,6,7,8) # вызван инициализатор из класса Rect
print(r.__dict__)

инициализация Геом для <class '__main__.Line'>
инициализатор Rect
{'fill': None}


Но в данном случае у нас в экземпляре r не создались атрибуты-координаты. Это можно сделать, явно вызвав в классе Rect инициализатор его базового класса

In [14]:
class Geom:
    def __init__(self, x1, y1, x2, y2):
        print(f'инициализация Геом для {self.__class__}')
        self.x1 = x1
        self.y1 = y1
        self.x2 = x2
        self.y2 = y2

class Line(Geom):
    pass

class Rect(Geom):
    def __init__(self, x1, y1, x2, y2, fill=None):
        # Geom.__init__(self, x1, y1, x2, y2) # Все работает, однако тут мы явно указали имя базового класса, из которого вызываем метод __init__
        super().__init__(x1, y1, x2, y2)
        print('инициализатор Rect')
        self.fill = fill

l = Line(1,2,3,4) # вызыван инициализатор из базового класса
r = Rect(5,6,7,8) # вызван инициализатор из класса Rect
print(r.__dict__)

инициализация Геом для <class '__main__.Line'>
инициализация Геом для <class '__main__.Rect'>
инициализатор Rect
{'x1': 5, 'y1': 6, 'x2': 7, 'y2': 8, 'fill': None}


Функция super() используется для обращения к базовому классу. Она возвращает ссылку на объект-посредник, через который происходит обращение к базовому классу. А раз объект, то для него self прописывать уже не нужно. Вызов методов базового класса через функцию super() называется делегированием